# Bronze | Schedules

## Purpose

This notebook ingests NFL schedule data from **nflverse** into the Bronze layer of the NFL Data Intelligence Platform.

The source dataset contains historical and scheduled NFL games, including information such as:

- Game ID
- Season
- Week
- Game date
- Home and away teams
- Scores
- Rest days
- Betting information
- Weather conditions
- Stadium information
- Quarterback information

## Source

The data is obtained from the official **nflverse-data** repository through an HTTP request.

Source file:

`games.csv`

Source:

`https://github.com/nflverse/nflverse-data/releases/download/schedules/games.csv`

## Pipeline

```text
nflverse
    │
    ▼
HTTP Request
    │
    ▼
Landing Zone
    │
    ▼
Spark DataFrame
    │
    ▼
Data Quality
    │
    ▼
Ingestion Metadata
    │
    ▼
Delta Table
    │
    ▼
nfl_bronze.schedules

In [0]:
%python

# =====================================================================
# NFL Data Intelligence Platform
# Bronze Layer - Schedules
# =====================================================================
# Configuration
# =====================================================================

import requests
import uuid
import time
from datetime import datetime, timezone
from io import BytesIO

# ---------------------------------------------------------------------
# Source
# ---------------------------------------------------------------------

SOURCE_NAME = "nflverse"
SOURCE_FILE = "games.csv"

SOURCE_URL = (
    "https://github.com/nflverse/nflverse-data/"
    "releases/download/schedules/games.csv"
)

# ---------------------------------------------------------------------
# Landing Zone
# ---------------------------------------------------------------------

LANDING_PATH = (
    "/Volumes/workspace/default/landing_zone/games.csv"
)

# ---------------------------------------------------------------------
# Bronze Table
# ---------------------------------------------------------------------

BRONZE_TABLE = "nfl_bronze.schedules"

# ---------------------------------------------------------------------
# Ingestion Run
# ---------------------------------------------------------------------

RUN_ID = str(uuid.uuid4())

START_TIME = datetime.now(timezone.utc)

print("Run ID:", RUN_ID)
print("Start time:", START_TIME)
print("Source:", SOURCE_URL)
print("Landing file:", LANDING_PATH)
print("Bronze table:", BRONZE_TABLE)

## 2. Source Download and File Replacement

This section downloads the latest `games.csv` file from nflverse and stores it in the Unity Catalog Volume used as the project's Landing Zone.

Before replacing the existing file, the HTTP response is validated.

The process follows this order:

```text
HTTP Request
     │
     ▼
Validate HTTP Status
     │
     ▼
Validate File Content
     │
     ▼
Check Existing games.csv
     │
     ├── Exists → Delete previous file
     │
     └── Does not exist → Continue
     │
     ▼
Write new games.csv

In [0]:
# =====================================================================
# Download source file
# =====================================================================

try:

    print("Starting download...")
    
    response = requests.get(
        SOURCE_URL,
        timeout=60
    )

    # -------------------------------------------------------------
    # Validate HTTP response
    # -------------------------------------------------------------

    if response.status_code != 200:
        raise RuntimeError(
            f"HTTP request failed. "
            f"Status code: {response.status_code}"
        )

    # -------------------------------------------------------------
    # Validate content
    # -------------------------------------------------------------

    if not response.content:
        raise RuntimeError(
            "Downloaded file is empty."
        )

    print("HTTP status:", response.status_code)
    print("Downloaded bytes:", len(response.content))

    # -------------------------------------------------------------
    # Remove previous file if it exists
    # -------------------------------------------------------------

    landing_directory = (
        "/Volumes/workspace/default/landing_zone"
    )

    existing_files = dbutils.fs.ls(landing_directory)

    file_exists = any(
        file.name == SOURCE_FILE
        for file in existing_files
    )

    if file_exists:

        print(f"Existing file found: {SOURCE_FILE}")
        print("Removing previous file...")

        dbutils.fs.rm(LANDING_PATH)

        print("Previous file removed.")

    else:

        print("No previous games.csv found.")

    # -------------------------------------------------------------
    # Write new file
    # -------------------------------------------------------------

    with open(LANDING_PATH, "wb") as file:
        file.write(response.content)

    print(f"New file successfully written: {LANDING_PATH}")

except Exception as e:

    print("ERROR during source ingestion.")
    print("Error type:", type(e).__name__)
    print("Error message:", str(e))

    raise

## 3. Read Landing File with Spark

After the source file has been stored in the Landing Zone, it is read using Apache Spark.

Spark is used as the main processing engine because the project is being developed as a Data Engineering platform in Databricks.

The CSV reader is configured to:

- Use the first row as column headers.
- Infer the data types from the source data.
- Load the file into a Spark DataFrame.

The resulting DataFrame is used by the following stages of the Bronze pipeline.

```text
Landing Zone
     │
     ▼
Spark CSV Reader
     │
     ▼
Spark DataFrame

In [0]:
# =====================================================================
# Read landing file with Spark
# =====================================================================

try:

    print("Reading landing file with Spark...")

    df_spark = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(LANDING_PATH)
    )

    row_count = df_spark.count()
    column_count = len(df_spark.columns)

    print("Rows:", row_count)
    print("Columns:", column_count)

    df_spark.printSchema()

except Exception as e:

    print("ERROR reading landing file.")
    print("Error type:", type(e).__name__)
    print("Error message:", str(e))

    raise

## 4. Data Quality Checks

Before writing the dataset to the Bronze table, basic data quality checks are performed.

The checks validate:

- Required columns exist.
- The dataset contains records.
- `game_id` is not NULL.
- `game_id` values are unique.
- Home and away team values are not NULL.

These checks are intentionally limited at the Bronze stage.

Bronze should not apply business transformations or complex analytical rules. Its main responsibility is to verify that the incoming source is structurally usable before being stored.

For example, scores are not required to be non-null because the source can contain games that have not yet been played.

In [0]:
# =====================================================================
# Data Quality Checks
# =====================================================================

try:

    print("Starting data quality checks...")

    # -------------------------------------------------------------
    # Expected critical columns
    # -------------------------------------------------------------

    required_columns = [
        "game_id",
        "season",
        "week",
        "gameday",
        "away_team",
        "home_team"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df_spark.columns
    ]

    if missing_columns:

        raise ValueError(
            f"Missing required columns: {missing_columns}"
        )

    print("Required columns: PASS")

    # -------------------------------------------------------------
    # Row count
    # -------------------------------------------------------------

    if row_count == 0:

        raise ValueError(
            "Data quality check failed: dataset contains 0 rows."
        )

    print(f"Row count: PASS ({row_count} rows)")

    # -------------------------------------------------------------
    # NULL game_id
    # -------------------------------------------------------------

    null_game_id_count = (
        df_spark
        .filter("game_id IS NULL")
        .count()
    )

    if null_game_id_count > 0:

        raise ValueError(
            f"Data quality check failed: "
            f"{null_game_id_count} rows have NULL game_id."
        )

    print("game_id NULL check: PASS")

    # -------------------------------------------------------------
    # Duplicate game_id
    # -------------------------------------------------------------

    duplicate_game_id_count = (
        df_spark
        .groupBy("game_id")
        .count()
        .filter("count > 1")
        .count()
    )

    if duplicate_game_id_count > 0:

        raise ValueError(
            f"Data quality check failed: "
            f"{duplicate_game_id_count} duplicated game_id values."
        )

    print("game_id uniqueness: PASS")

    # -------------------------------------------------------------
    # NULL critical team fields
    # -------------------------------------------------------------

    null_team_count = (
        df_spark
        .filter(
            """
            away_team IS NULL
            OR home_team IS NULL
            """
        )
        .count()
    )

    if null_team_count > 0:

        raise ValueError(
            f"Data quality check failed: "
            f"{null_team_count} rows have NULL team values."
        )

    print("Team fields NULL check: PASS")

    print("All data quality checks passed.")

except Exception as e:

    print("ERROR during data quality validation.")
    print("Error type:", type(e).__name__)
    print("Error message:", str(e))

    raise

## 5. Ingestion Metadata

This section adds metadata describing how and when the source data was ingested.

The following metadata columns are added:

- `ingestion_timestamp`: UTC timestamp of the ingestion.
- `ingestion_run_id`: Unique identifier for the execution.
- `source`: Name of the source system.
- `source_file`: Name of the ingested file.
- `source_url`: URL used to obtain the source data.

This metadata improves traceability and allows individual records to be associated with a specific ingestion execution.

```text
Source Data
     │
     ├── Original columns
     │
     └── Ingestion metadata
              │
              ▼
        Bronze DataFrame

In [0]:
# =====================================================================
# Add ingestion metadata
# =====================================================================

from pyspark.sql.functions import (
    current_timestamp,
    lit
)

INGESTION_TIMESTAMP = datetime.now(timezone.utc)

df_bronze = (
    df_spark
    .withColumn(
        "ingestion_timestamp",
        lit(INGESTION_TIMESTAMP)
    )
    .withColumn(
        "ingestion_run_id",
        lit(RUN_ID)
    )
    .withColumn(
        "source",
        lit(SOURCE_NAME)
    )
    .withColumn(
        "source_file",
        lit(SOURCE_FILE)
    )
    .withColumn(
        "source_url",
        lit(SOURCE_URL)
    )
)

print("Metadata columns added.")

display(df_bronze.limit(10))

## 6. Write Bronze Delta Table

The validated DataFrame is written to the Bronze layer as a Delta table.

Target table:

`nfl_bronze.schedules`

The table is written using `overwrite` mode because the source `games.csv` represents a complete snapshot of the schedules dataset.

Each execution replaces the previous Bronze snapshot with the latest successfully validated source data.

```text
Validated DataFrame
        │
        ▼
     Delta
        │
        ▼
nfl_bronze.schedules

In [0]:
# =====================================================================
# Write Bronze Delta table
# =====================================================================

try:

    print("Writing Bronze table...")

    (
        df_bronze
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(BRONZE_TABLE)
    )

    print(
        f"Bronze table successfully written: {BRONZE_TABLE}"
    )

except Exception as e:

    print("ERROR writing Bronze table.")
    print("Error type:", type(e).__name__)
    print("Error message:", str(e))

    raise

## 7. Execution Metrics

This section calculates basic execution metrics for the ingestion process.

The metrics include:

- Run ID
- Source
- Source file
- Number of rows loaded
- Start time
- End time
- Total execution duration
- Execution status

These metrics provide basic operational visibility into the ingestion process.

Example:

```text
INGESTION SUMMARY

Source: nflverse
File: games.csv
Rows loaded: 7548
Start time: ...
End time: ...
Duration: ...
Status: SUCCESS

In [0]:
# =====================================================================
# Execution metrics
# =====================================================================

END_TIME = datetime.now(timezone.utc)

duration_seconds = (
    END_TIME - START_TIME
).total_seconds()

rows_loaded = df_bronze.count()

print("==============================================")
print("INGESTION SUMMARY")
print("==============================================")
print("Run ID:", RUN_ID)
print("Source:", SOURCE_NAME)
print("File:", SOURCE_FILE)
print("Rows loaded:", rows_loaded)
print("Start time:", START_TIME)
print("End time:", END_TIME)
print("Duration seconds:", duration_seconds)
print("Status: SUCCESS")
print("==============================================")